# Career Intelligence Agent — Dev & Troubleshooting Notebook

## Setup

In [1]:
import os
import json
from dotenv import load_dotenv
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage
from lib.tools import (
    get_current_time,
    web_search,
    google_docs_create,
    google_docs_read,
    google_docs_write,
    google_docs_append,
    google_docs_replace_text,
    google_sheets_read,
    google_sheets_column,
    google_sheets_append,
    google_sheets_update_cell,
    google_sheets_find_row,
    gmail_read,
    gmail_send,
    gmail_create_draft,
    jsearch_request,
)
from lib.prompts import (
    ORCHESTRATOR_PROMPT,
    JOB_RESEARCHER_PROMPT,
    SCORER_ANALYST_PROMPT,
    REPORT_WRITER_PROMPT,
    INTERVIEW_COACH_PROMPT,
    JOB_CATALOGUER_PROMPT,
    CRITIC_PROMPT,
)
from main import _cached, SUBAGENTS, build_agents

load_dotenv()
print("ANTHROPIC_API_KEY:          ", "set" if os.getenv("ANTHROPIC_API_KEY") else "MISSING")
print("OPENAI_API_KEY:             ", "set" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("LANGSMITH_API_KEY:          ", "set" if os.getenv("LANGSMITH_API_KEY") else "MISSING")
print("JSEARCH_API_KEY:            ", "set" if os.getenv("JSEARCH_API_KEY") else "MISSING")
print("TAVILY_API_KEY:             ", "set" if os.getenv("TAVILY_API_KEY") else "MISSING")
print("GOOGLE_SERVICE_ACCOUNT_PATH:", os.getenv("GOOGLE_SERVICE_ACCOUNT_PATH", "service_account.json (default)"))
print("GMAIL_CLIENT_SECRETS_PATH:  ", os.getenv("GMAIL_CLIENT_SECRETS_PATH", "gmail_credentials.json (default)"))
print("GMAIL_TOKEN_PATH:           ", os.getenv("GMAIL_TOKEN_PATH", "gmail_token.json (default)"))

/Users/ericlivingston/Documents/Projects/career_intelligence_agent/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ANTHROPIC_API_KEY:           set
OPENAI_API_KEY:              set
LANGSMITH_API_KEY:           set
JSEARCH_API_KEY:             set
TAVILY_API_KEY:              set
GOOGLE_SERVICE_ACCOUNT_PATH: service_account.json (default)
GMAIL_CLIENT_SECRETS_PATH:   gmail_credentials.json (default)
GMAIL_TOKEN_PATH:            gmail_token.json (default)


## Agent Builder

In [2]:
def build_subagent(name: str):
    """Build a single subagent in isolation for direct testing."""
    cfg = next((a for a in SUBAGENTS if a["name"] == name), None)
    if cfg is None:
        raise ValueError(f"Unknown subagent: {name!r}. Options: {[a['name'] for a in SUBAGENTS]}")
    return create_deep_agent(
        name=cfg["name"],
        model=cfg["model"],
        tools=cfg["tools"],
        system_prompt=cfg["system_prompt"],
        backend=FilesystemBackend(root_dir=".", virtual_mode=True),
        checkpointer=MemorySaver(),
    )

print("build_agents() and build_subagent(name) ready.")
print("Subagents:", [a["name"] for a in SUBAGENTS])

build_agents() and build_subagent(name) ready.
Subagents: ['job_researcher', 'scorer_analyst', 'report_writer', 'interview_coach', 'job_cataloguer', 'critic_agent']


## Invoke Helpers

In [3]:
def invoke(agent, message: str, thread_id: str = "dev-session") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config=config,
    )
    messages = result.get("messages", [])
    response = messages[-1].content if messages else ""
    print(response)
    return response


def invoke_json(agent, payload: dict, thread_id: str = "dev-session") -> str:
    """Send a structured JSON payload — useful for testing sub-agents directly."""
    return invoke(agent, json.dumps(payload, indent=2), thread_id=thread_id)


print("Helpers ready: invoke(agent, message), invoke_json(agent, payload)")

Helpers ready: invoke(agent, message), invoke_json(agent, payload)


---
## Test: Google Sheets Tools

In [8]:
# Set your spreadsheet ID and sheet name here
SPREADSHEET_ID = "1wlXCsRVph7DKB_Jv0MaYlji9p_aUIUlC3QYfspJQMCc"   # paste your Google Sheet ID
SHEET_NAME = "job_catalogue" # update if your tab has a different name

In [9]:
# Read all rows — full read (app_tracker use case); confirms output is compact valid JSON
result = google_sheets_read.invoke({"spreadsheet_id": SPREADSHEET_ID, "sheet_name": SHEET_NAME})
rows = json.loads(result)  # was Python repr before — assert it parses cleanly
print(f"Rows returned: {len(rows)}")
print(f"Keys per row : {list(rows[0].keys()) if rows else '[]'}")
print(result[:400])  # preview first 400 chars


In [ ]:
# google_sheets_column — single-column lookup (job_researcher duplicate-check use case)
# Uses job_id (column A) — the canonical dedup key matched against jsearch results
result = google_sheets_column.invoke({
    "spreadsheet_id": SPREADSHEET_ID,
    "sheet_name": SHEET_NAME,
    "column": "job_id",
})
values = json.loads(result)
print(f"google_sheets_column → {len(values)} values returned")
print(f"Type  : {type(values)}")
print(f"Sample: {[v for v in values if v][:3]}")
assert isinstance(values, list), "Expected a JSON array"
print("PASS")

In [10]:
# Append a test row (matches the app_tracker tracking schema columns A–Q)
test_row = [
    "test-job-001",      # A: job_id
    "Hardware Engineer", # B: title
    "Anduril Industries",# C: company
    "Costa Mesa, CA",    # D: location
    "Hybrid",            # E: remote_status
    "$130k–$180k",       # F: salary_range
    "https://example.com/job", # G: posting_url
    "LinkedIn",          # H: source
    "2026-05-06",        # I: date_discovered
    "",                  # J: date_applied
    "discovered",        # K: status
    "",                  # L: composite_score
    "",                  # M: recommendation
    "Dev test row",      # N: notes
    "",                  # O: next_action
    "",                  # P: next_action_date
    "2026-05-06",        # Q: last_updated
]

result = google_sheets_append.invoke({"spreadsheet_id": SPREADSHEET_ID, "sheet_name": SHEET_NAME, "row": test_row})
print(result)

Row appended successfully.


In [11]:
# Find a row by value (e.g. search by job_id or company name)
result = google_sheets_find_row.invoke({"spreadsheet_id": SPREADSHEET_ID, "sheet_name": SHEET_NAME, "query": "test-job-001"})
print(result)

{'row_index': 26, 'values': ['test-job-001', 'Hardware Engineer', 'Anduril Industries', 'Costa Mesa, CA', 'Hybrid', '$130k–$180k', 'https://example.com/job', 'LinkedIn', '2026-05-06', '', 'discovered', '', '', 'Dev test row', '', '', '2026-05-06']}


In [12]:
# Update a single cell — change the status of the test row
# First find the row index from the find_row result above, then update column K (index 11)
found = eval(result)  # convert string repr back to dict
row_index = found["row_index"]

update_result = google_sheets_update_cell.invoke({
    "spreadsheet_id": SPREADSHEET_ID,
    "sheet_name": SHEET_NAME,
    "row_index": row_index,
    "col_index": 11,  # K: status
    "value": "shortlisted",
})
print(update_result)

Cell (26, 11) updated to 'shortlisted'.


---
## Test: Google Docs Tools

In [4]:
# Create a new Google Doc — google_docs_create
result = google_docs_create.invoke({"title": "Dev Test Doc — career intelligence agent"})
print(result)

# Auto-populate DOCUMENT_ID so the cells below can use the new doc
doc_info = json.loads(result)
DOCUMENT_ID = doc_info["document_id"]
print("DOCUMENT_ID set to:", DOCUMENT_ID)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=495833492794-r2soghhlg7luob4diavp7v2fsclk2qt0.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A58894%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.compose+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocuments+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=1l5vTWNVvvQFVWlHKfVj5NPeviF1eQ&code_challenge=oyB6uYZy-TFP95oFKxfsrbXrCffu2IvQRRePCbSascI&code_challenge_method=S256&access_type=offline
{"document_id": "1-CGqZ-a2ou30aM2XZclL_OIJseRQ7ClBHENb-8Ae444", "doc_url": "https://docs.google.com/document/d/1-CGqZ-a2ou30aM2XZclL_OIJseRQ7ClBHENb-8Ae444/edit"}
DOCUMENT_ID set to: 1-CGqZ-a2ou30aM2XZclL_OIJseRQ7ClBHENb-8Ae444


In [3]:
# Set your Google Doc ID here (from the URL: /document/d/DOC_ID/edit)
DOCUMENT_ID = "13JFDlQp547jGLqBFQgc6nyyuZu41UXz7kGDAYOhMaOE"  # paste your Google Doc ID

In [8]:
# Read the full text of the document
result = google_docs_read.invoke({"document_id": DOCUMENT_ID})
print(result)

This is a test
Nani is cute
Nani is kind
Nani is reading something on her computer
Nani is a silly little goose
Nani is bored right now



In [4]:
# Append a test line to the end of the document
result = google_docs_append.invoke({
    "document_id": DOCUMENT_ID,
    "text": "\nDev test append — career intelligence agent",
})
print(result)

Text appended successfully.


In [5]:
# Replace the test text we just appended (verifies find-and-replace works)
result = google_docs_replace_text.invoke({
    "document_id": DOCUMENT_ID,
    "find": "Dev test append — career intelligence agent",
    "replacement": "Dev test replace — career intelligence agent ✓",
})
print(result)

Replaced 1 occurrence(s) of 'Dev test append — career intelligence agent'.


---
## Test: Gmail Tools

> **First run:** a browser window will open asking you to authorize Gmail access.  
> The token is saved to `gmail_token.json` and reused on subsequent runs.  
> Requires `gmail_credentials.json` (OAuth 2.0 Desktop client secrets) in the project root.

In [3]:
# gmail_read — fetch the 5 most recent emails (no query filter)
result = gmail_read.invoke({"query": "", "max_results": 5})
messages = json.loads(result)
for msg in messages:
    print(f"[{msg['date']}] {msg['from']}")
    print(f"  Subject : {msg['subject']}")
    print(f"  Snippet : {msg['snippet']}")
    print()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=495833492794-r2soghhlg7luob4diavp7v2fsclk2qt0.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49746%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.compose&state=Tkes53FPFuyqvUrQUwKCR8Mfi8xNai&code_challenge=mErqMHyVOAKJRBKk85yEAKuVtEBeBy1UG-GIe0PEJQY&code_challenge_method=S256&access_type=offline
[Mon, 11 May 2026 22:52:06 -0700] Eric Livingston <ericlivingston0@gmail.com>
  Subject : Re: Daily Digest — April 29, 2026
  Snippet : Thank you, this is a test On Wed, Apr 29, 2026 at 9:25 PM &lt;researchagent69@gmail.com&gt; wrote: 📋 Daily Digest — Wednesday, April 29, 2026 Searched 20 recent postings for Electrical Engineer and

[Thu, 07 May 2026 04:21:14 GMT] Google <no-reply@accounts.google.com>
  Subject : Your Google Account was 

In [4]:
# gmail_read — search with a query (e.g. unread emails or by keyword)
result = gmail_read.invoke({"query": "is:unread", "max_results": 3})
unread = json.loads(result)
print(f"Unread messages found: {len(unread)}")
for msg in unread:
    print(f"  [{msg['date']}] {msg['subject']} — from {msg['from']}")

Unread messages found: 3
  [Mon, 11 May 2026 22:52:06 -0700] Re: Daily Digest — April 29, 2026 — from Eric Livingston <ericlivingston0@gmail.com>
  [Thu, 07 May 2026 04:21:14 GMT] Your Google Account was recovered successfully — from Google <no-reply@accounts.google.com>
  [Thu, 07 May 2026 04:21:10 GMT] Security alert — from Google <no-reply@accounts.google.com>


In [5]:
# gmail_create_draft — create a draft without sending
# Update DRAFT_TO to your own address so you can verify it in Gmail > Drafts
DRAFT_TO = "ericlivingston0@gmail.com"

result = gmail_create_draft.invoke({
    "to": DRAFT_TO,
    "subject": "Dev test draft — career intelligence agent",
    "body": "This is an automated draft created by the gmail_create_draft tool.\nSafe to delete.",
})
print(result)
DRAFT_ID = result.split("id: ")[-1].strip()
print(f"Draft ID captured: {DRAFT_ID}")

Draft created with id: r-8026722722693315040
Draft ID captured: r-8026722722693315040


In [6]:
# gmail_send — send an email directly
# Sending to yourself is the safest way to verify end-to-end delivery
SEND_TO = "ericlivingston0@gmail.com"

result = gmail_send.invoke({
    "to": SEND_TO,
    "subject": "Dev test send — career intelligence agent",
    "body": "This is an automated send test from the gmail_send tool.\nSafe to delete.",
})
print(result)

# Verify it landed by reading recent mail filtered by the test subject
import time; time.sleep(2)  # brief pause for delivery
verify = gmail_read.invoke({"query": "subject:Dev test send", "max_results": 1})
found = json.loads(verify)
if found:
    print(f"Verified in inbox: [{found[0]['date']}] {found[0]['subject']}")
else:
    print("Not yet visible via API — check Gmail directly if needed.")

Email sent. Message id: 19e1ac379f8e7f6d
Verified in inbox: [Tue, 12 May 2026 00:57:58 -0500] Dev test send — career intelligence agent


In [ ]:
# jsearch_request (search) — output is compact JSON (no indentation) to reduce token usage
result = jsearch_request.invoke({
    "endpoint": "search",
    "query": "hardware engineer San Diego",
    "page": 1,
    "num_pages": 1,
    "date_posted": "week",
    "remote_jobs_only": False,
    "country": "us",
})
assert '\n' not in result, "Output should be compact JSON (no newlines)"
data = json.loads(result)

jobs = data.get("data", data) if isinstance(data, dict) else data
print(f"jsearch_request (search) → {len(jobs)} job(s) returned")
for job in jobs[:3]:
    print(f"  [{job.get('job_id', '?')[:20]}...]  {job.get('job_title')} @ {job.get('employer_name')}")
    print(f"    Location    : {job.get('job_city')}, {job.get('job_state')}")
    print(f"    Posted      : {job.get('job_posted_at_datetime_utc', 'n/a')[:10]}")
    print(f"    Description : {job.get('job_description', '')[:120].strip()}...")
    print()

assert isinstance(jobs, list), "Expected a list of job results"
assert len(jobs) > 0, "No jobs returned — check API key and query"
assert len(jobs[0].get("job_description", "")) < 2000, "Descriptions should be summarized to ~50-100 words"

SAMPLE_JOB_ID = jobs[0].get("job_id")
print(f"Sample job_id captured: {SAMPLE_JOB_ID}")
print("PASS")


In [ ]:
# jsearch_request (job-details) — fetch full details for a job_id from the search above
# Run the search cell first to populate SAMPLE_JOB_ID
result = jsearch_request.invoke({
    "endpoint": "job-details",
    "job_id": SAMPLE_JOB_ID,
    "extended_publisher_details": False,
})
data = json.loads(result)

detail = data.get("data", data)
if isinstance(detail, list):
    detail = detail[0]

print("jsearch_request (job-details)")
print(f"  job_id          : {detail.get('job_id', '?')[:40]}...")
print(f"  title           : {detail.get('job_title')}")
print(f"  employer        : {detail.get('employer_name')}")
print(f"  location        : {detail.get('job_city')}, {detail.get('job_state')}")
print(f"  employment_type : {detail.get('job_employment_type')}")
print(f"  salary_min      : {detail.get('job_min_salary')}")
print(f"  salary_max      : {detail.get('job_max_salary')}")
print(f"  apply_url       : {str(detail.get('job_apply_link', 'n/a'))[:60]}...")
desc = detail.get("job_description", "")
print(f"  description     : {desc[:120].strip()}...")

assert detail.get("job_id") == SAMPLE_JOB_ID, "Returned job_id should match the requested id"
print("PASS")

---
## Test: JSearch Tool

`jsearch_request` routes to either the search or job-details endpoint of the JSearch API via an `endpoint` parameter.  
Search results have their `job_description` fields replaced with a 50–100 word GPT-4.1-nano summary before returning. Both endpoints return **compact JSON** (no indentation) to reduce token consumption downstream.  
Both endpoints read `JSEARCH_API_KEY` from the environment automatically.

> **Requires** `JSEARCH_API_KEY` and `OPENAI_API_KEY` set in `.env`. Cells will raise an HTTP 401/403 if either key is missing or invalid.  
> Run the **search** cell first — it captures `SAMPLE_JOB_ID` used by the **job-details** cell.


---
## Test: Tavily Web Search

In [4]:
# web_search — basic query
# Verifies the Tavily client initializes and returns results
result = web_search.invoke({"query": "Anduril Industries Glassdoor reviews 2026", "max_results": 3})
parsed = json.loads(result)
print(f"Results returned: {len(parsed)}")
for r in parsed:
    print(f"  [{r['title']}]({r['url']})")
    print(f"  {r['content'][:120]}...\n")

Results returned: 3
  [Anduril Reviews in Lexington - Glassdoor](https://www.glassdoor.com/Reviews/Anduril-Lexington-Reviews-EI_IE3546800.0,7_IL.8,17_IC1154599.htm)
  The company has very nice unlimited perks, has a very fast pace on product development, and ultra-high technological lev...

  [Working at Anduril - Glassdoor](https://www.glassdoor.com/Overview/Working-at-Anduril-EI_IE3546800.11,18.htm)
  Anduril has an employee rating of 3.9 out of 5 stars, based on 273 company reviews on Glassdoor which indicates that mos...

  [Just a moment...](https://www.glassdoor.com/Reviews/Anduril-long-hour-Reviews-EI_IE3546800.0,7_KH8,17.htm)
  Glassdoor est fondé sur les contributions de véritables employés et chercheurs d’emploi. Nous utilisons des systèmes de ...



In [ ]:
# web_search — salary / compensation query (scorer_analyst use case)
result = web_search.invoke({"query": "Hardware Engineer salary San Diego 2026", "max_results": 5})
parsed = json.loads(result)
print(f"Results returned: {len(parsed)}")
for r in parsed:
    print(f"  [{r['title']}] — {r['content'][:150]}\n")

In [ ]:
# web_search — company news query (report_writer / critic use case)
result = web_search.invoke({"query": "Anduril Industries funding news layoffs 2026"})
parsed = json.loads(result)
print(f"Results returned: {len(parsed)}")
for r in parsed:
    print(f"  [{r['title']}]\n  {r['url']}\n  {r['content'][:200]}\n")

---
## Test: Full Orchestrator

In [4]:
orchestrator = build_agents()

In [5]:
invoke(orchestrator, "Log 5 test jobs in the job_catalogue")

Skill 'output-format' in /skills/output_format/SKILL.md does not follow Agent Skills specification: name 'output-format' must match directory name 'output_format'. Consider renaming for spec compliance.


Done. All 5 test jobs are now logged in the job_catalogue with their respective statuses, dates, and notes. You can query or update them anytime using the job_cataloguer.


'Done. All 5 test jobs are now logged in the job_catalogue with their respective statuses, dates, and notes. You can query or update them anytime using the job_cataloguer.'

---
## Test: Sub-Agents in Isolation

### Job Researcher

In [6]:
researcher = build_subagent("job_researcher")

invoke_json(researcher, {
    "criteria": "Electrical Engineer",
    "task": "Find recent hardware engineering roles at Anduril and Northrop Grumman",
    "resume_summary": "3 years experience in PCB design and embedded firmware, BS EE",
    "spreadsheet_id": SPREADSHEET_ID,
    "sheet_name": SHEET_NAME,
})

Now filtering against the catalogue. Extracting only relevant hardware engineering roles that match the criteria (Electrical Engineer background) and aren't already in the sheet.

Excluded from catalogue check:
- Principal/Sr Principal roles (requires 15+ years, user has 3)
- FPGA Engineer / Firmware roles (not hardware-focused)
- Manufacturing Engineer (procurement/supply chain)
- V&V Systems Engineer (testing role, not design)

Filtered results for 3-year career level:

```json
[
  {
    "job_id": "JAn9UOFmyZZ6Tiv8AAAAAA==",
    "job_title": "Electrical Hardware Engineer Jobs",
    "employer_name": "Anduril Industries",
    "job_publisher": "Clearance Jobs",
    "job_employment_type": "Full-time",
    "job_posted_at": "2026-05-04T00:00:00.000Z",
    "job_city": "Lexington",
    "job_salary": null,
    "job_min_salary": 129000,
    "job_max_salary": 171000,
    "job_apply_link": "https://www.clearancejobs.com/jobs/8894291/electrical-hardware-engineer",
    "job_description": "Anduril 

'Now filtering against the catalogue. Extracting only relevant hardware engineering roles that match the criteria (Electrical Engineer background) and aren\'t already in the sheet.\n\nExcluded from catalogue check:\n- Principal/Sr Principal roles (requires 15+ years, user has 3)\n- FPGA Engineer / Firmware roles (not hardware-focused)\n- Manufacturing Engineer (procurement/supply chain)\n- V&V Systems Engineer (testing role, not design)\n\nFiltered results for 3-year career level:\n\n```json\n[\n  {\n    "job_id": "JAn9UOFmyZZ6Tiv8AAAAAA==",\n    "job_title": "Electrical Hardware Engineer Jobs",\n    "employer_name": "Anduril Industries",\n    "job_publisher": "Clearance Jobs",\n    "job_employment_type": "Full-time",\n    "job_posted_at": "2026-05-04T00:00:00.000Z",\n    "job_city": "Lexington",\n    "job_salary": null,\n    "job_min_salary": 129000,\n    "job_max_salary": 171000,\n    "job_apply_link": "https://www.clearancejobs.com/jobs/8894291/electrical-hardware-engineer",\n    "j

### Scorer / Analyst

In [5]:
scorer = build_subagent("scorer_analyst")

invoke_json(scorer, {
    "jobs": [
        {
            "job_id": "test-001",
            "job_title": "Hardware Engineer",
            "employer_name": "Anduril Industries",
            "job_city": "Costa Mesa, CA",
            "job_min_salary": 130000,
            "job_max_salary": 180000,
            "job_description": "Design and develop hardware systems for autonomous defense platforms.",
        }
    ],
    "resume": "3 years PCB design, embedded C, FPGA prototyping. BS EE from UCSD.",
    "principles": "Wants mission-driven work, target comp $150k+, prefers small teams.",
    "scoring_rubric": "culture_sentiment 0.30, compensation 0.25, skill_alignment 0.20, growth_opportunity 0.15, company_health 0.10",
})

[]


[]

### Report Writer

In [4]:
report_writer = build_subagent("report_writer")

invoke_json(report_writer, {
    "subject": "general atomics",
    "report_type": "company_profile",
    "resume_summary": "3 years PCB design, embedded C, FPGA prototyping. BS EE from UCSD.",
    "principles": "Wants mission-driven work, target comp $150k+, prefers small teams.",
    "scored_job": None,
})

Report is written. Here's the summary:

```json
{
  "doc_url": "https://docs.google.com/document/d/1TJ4ANo9qXR4JisxBfFAkBlMJna9skdO-t9MRAVNiCrg/edit",
  "report_type": "company_profile",
  "subject": "General Atomics",
  "key_finding": "GA is a financially stable, mission-driven defense contractor with strong fit for your PCB/FPGA/embedded skillset, but base salaries at 3 years experience ($108K–$125K for FPGA) fall short of your $150K target and the company offers no equity as a private firm.",
  "recommendation": "Target GA-ASI (Poway) or GA-EMS (San Diego) specifically and negotiate toward senior FPGA roles — $150K+ is achievable but requires deliberate role selection and strong negotiation, not a standard outcome at this experience level."
}
```

**[View the full report →](https://docs.google.com/document/d/1TJ4ANo9qXR4JisxBfFAkBlMJna9skdO-t9MRAVNiCrg/edit)**

---

**Bottom line:** GA is a legitimate mission-driven target for your background, but go in with eyes open on two things:

'Report is written. Here\'s the summary:\n\n```json\n{\n  "doc_url": "https://docs.google.com/document/d/1TJ4ANo9qXR4JisxBfFAkBlMJna9skdO-t9MRAVNiCrg/edit",\n  "report_type": "company_profile",\n  "subject": "General Atomics",\n  "key_finding": "GA is a financially stable, mission-driven defense contractor with strong fit for your PCB/FPGA/embedded skillset, but base salaries at 3 years experience ($108K–$125K for FPGA) fall short of your $150K target and the company offers no equity as a private firm.",\n  "recommendation": "Target GA-ASI (Poway) or GA-EMS (San Diego) specifically and negotiate toward senior FPGA roles — $150K+ is achievable but requires deliberate role selection and strong negotiation, not a standard outcome at this experience level."\n}\n```\n\n**[View the full report →](https://docs.google.com/document/d/1TJ4ANo9qXR4JisxBfFAkBlMJna9skdO-t9MRAVNiCrg/edit)**\n\n---\n\n**Bottom line:** GA is a legitimate mission-driven target for your background, but go in with eyes o

### Interview Coach

In [ ]:
coach = build_subagent("interview_coach")

invoke_json(coach, {
    "role": "Hardware Engineer at Anduril Industries — autonomous defense systems",
    "resume": "3 years PCB design, embedded C, FPGA prototyping. BS EE from UCSD.",
    "request_type": "company_research_brief",
    "report": None,
    "scored_job": None,
})

### Job Cataloguer

In [ ]:
cataloguer = build_subagent("job_cataloguer")

invoke_json(cataloguer, {
    "action": "log_new",
    "job_data": {
        "title": "Hardware Engineer",
        "company": "Anduril Industries",
        "url": "https://jobs.lever.co/anduril/example",
        "date_applied": None,
        "status": "discovered",
        "notes": "High scorer, mission-aligned",
    },
    "spreadsheet_id": SPREADSHEET_ID,
    "sheet_name": SHEET_NAME,
})

### Critic Agent

In [ ]:
critic = build_subagent("critic_agent")

invoke_json(critic, {
    "source_agent": "scorer_analyst",
    "agent_output": {
        "job_id": "test-001",
        "company": "Anduril Industries",
        "composite_score": 84,
        "recommendation": "apply",
        "scores": {
            "culture_sentiment": {"score": 9, "rationale": "Glassdoor 4.5, mission-driven culture"},
            "compensation": {"score": 8, "rationale": "$130k-$180k meets target"},
            "skill_alignment": {"score": 7, "rationale": "PCB and embedded match well"},
            "growth_opportunity": {"score": 8, "rationale": "Rapidly growing defense tech"},
            "company_health": {"score": 9, "rationale": "$1.5B Series E, strong revenue"},
        },
    },
    "original_inputs": "Hardware Engineer role at Anduril, resume: PCB/embedded/FPGA, 3 yrs exp",
})

---
## Prompt Inspection

In [ ]:
PROMPTS = {
    "orchestrator": ORCHESTRATOR_PROMPT,
    "job_researcher": JOB_RESEARCHER_PROMPT,
    "scorer_analyst": SCORER_ANALYST_PROMPT,
    "report_writer": REPORT_WRITER_PROMPT,
    "interview_coach": INTERVIEW_COACH_PROMPT,
    "job_cataloguer": JOB_CATALOGUER_PROMPT,
    "critic_agent": CRITIC_PROMPT,
}

agent_name = "orchestrator"  # change to inspect a different agent
print(PROMPTS[agent_name])